In [ ]:
import os
from dotenv import load_dotenv
import io
import time
import base64
from pathlib import Path
from getpass import getpass

import pymupdf
import pandas as pd

from PIL import Image
from IPython.display import display

from groq import Groq

from pinecone import Pinecone, ServerlessSpec

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

In [ ]:
PDF_PATH = Path("data/2609.28504v1-2.pdf")

In [ ]:
TEXT_MODEL = "openai/gpt-oss-2b"
VISION_MODEL = "qwen/qwen3.8-27b"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

PINECONE_INDEX_NAME = "novacore-multimodal-rag"
PINECONE_NAMESPACE = "fy2026-demo"

IMAGE_DIR = Path("novacore_extracted_images")
IMAGE_DIR.mkdir(exist_ok=True)

print("PDF: ", PDF_PATH)

In [ ]:
## Adding GROQ and PINECONE API Keys
load_dotenv(".env")

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

text_llm = ChatGroq(
    model=TEXT_MODEL,
    temperature=0
)

In [ ]:
## Convert image to base64 for VLM Model
## Helper to convert image to base64

def image_to_data_uri(image_path):
    image_path = Path(image_path)

    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img.thumbnail((1600,1600))

        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=85)

    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{encoded}"


In [ ]:
## Helper:  Summarize images, charts and diagrams

def summarize_visual(image_path, page_number):
    image_data = image_to_data_uri(image_path)

    prompt = f"""
    This visual is extracted from page {page_number} of the AI insights paper.

    Describe the useful information visible in the visual.

    If it is a chart or graph:
    - mention important values
    - mention highest/lowest values
    - mention the main trend
    
    If it is a diagram:
    - identify important components
    - explain the flow or relationships

    If it is a normal image:
    - describe the usual factual information

    Keep the description concise and factual.
"""
    response = groq_client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {
                "role": "user",
                "content":[
                    {"type": "text", "text":prompt},
                    {
                        "type":"image_url",
                        "image_url": {"url": image_data},
                    }
                ]
            }
        ],
        temperature=0,
        max_completion_tokens=500
    )

    return response.choices[0].message.content.strip()

In [41]:
from langchain_core.documents import Document
## Extract text, tables and visuals from pdf

def extract_multimodal_documents(pdf_path):


    pdf = pymupdf.open(str(pdf_path))

    documents = []
    extracted_xrefs = set()


  
    for page_index in range(len(pdf)):
        page = pdf[page_index]
        page_number = page_index + 1

        print(f"Processing page {page_number}....")

        # -------------------------------------------------
        # TEXT Format Data Extraction to Langchain Document
        # -------------------------------------------------
        text = page.get_text("text").strip()
        if text:
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "page": page_number,
                        "modality": "text",
                        "source": pdf_path.name
                    }
                )
            )
        # -------------------------------------------------
        # TABLES Format Data Extraction to Langchain Document
        # -------------------------------------------------
        try:
            tables = page.find_tables().tables

            for table_number, table in enumerate(tables, start=1):
                df = table.to_pandas()

                if not df.empty:
                    table_text = df.to_markdown(index=False)

                    documents.append(
                        Document(
                            page_content=table_text,
                            metadata={
                                "page": page_number,
                                "modality": "table",
                                "table_number" : table_number,
                                "source": pdf_path.name
                            }
                        )
                    )
        except Exception as error:
            print("Table extractio warning: ", error)

        # ----------------------------------------------------------------------
        # IMAGES / CHARTS / DIAGRAMS Format Data Extraction to Langchain Document
        # -----------------------------------------------------------------------

        for image_number, image_info in enumerate(
            page.get_images(full=True),
            start=1
        ):
            xref = image_info[0]

            ## Do not process idential xref twice
            if xref in extracted_xrefs:
                continue

            extracted_xrefs.add(xref)

            image_data = pdf.extract_image(xref)
            image_bytes = image_data["image"]
            image_extension = image_data["ext"]

            image_path = (
                IMAGE_DIR
                / f"page_{page_number}_image_{image_number}.{image_extension}"
            )

            image_path.write_bytes(image_bytes)

            try:
                summary = summarize_visual(image_path, page_number)

                documents.append(
                    Document(
                        page_content=summary,
                        metadata={
                            "page": page_number,
                            "modality": "image",
                            "image_number": image_number,
                            "source": pdf_path.name
                        }
                    )
                )

            except Exception as error:
                print("Image extraction warning: ", error)

    pdf.close()
    return documents

documents = extract_multimodal_documents(PDF_PATH)

print()
print("Total Langchain Documents: ", len(documents))

Processing page 1....
Processing page 2....
Processing page 3....
Processing page 4....
Processing page 5....
Processing page 6....
Processing page 7....
Processing page 8....
Processing page 9....
Processing page 10....
Processing page 11....
Processing page 12....
Processing page 13....
Processing page 14....
Processing page 15....
Processing page 16....
Processing page 17....
Processing page 18....
Processing page 19....
Processing page 20....
Processing page 21....
Processing page 22....
Processing page 23....
Processing page 24....
Processing page 25....
Processing page 26....
Processing page 27....
Processing page 28....
Processing page 29....
Processing page 30....
Processing page 31....
Processing page 32....
Processing page 33....
Processing page 34....
Processing page 35....
Processing page 36....
Processing page 37....
Processing page 38....
Processing page 39....
Processing page 40....
Processing page 41....
Processing page 42....

Total Langchain Documents:  56


In [43]:
# Inspect extracted data

rows = []

for doc in documents:
    rows.append(
        {
            "page" : doc.metadata["page"],
            "modality" : doc.metadata["modality"],
            "preview": doc.page_content[:140].replace("\n", " ")
        }
    )

summary_df = pd.DataFrame(rows)

display(summary_df)

print("Document Modality")
display(summary_df["modality"].value_counts())

,page,modality,preview
0,1,text,AI in Science: Early Insights∗ Mihai Codreanu1...
1,2,text,1 Introduction Scientific discovery is one of ...
2,3,text,"2025; Appel et al., 2025; Handa et al., 2025; ..."
3,4,text,et al. (2026)) - to map scientific LLM interac...
4,5,text,documents. Specialized models are relatively m...
5,6,text,2 Data and Measurement 2.1 Preliminaries & Sco...
6,7,text,disciplines where research activities concentr...
7,8,text,their main job involves working directly in sc...
8,9,text,"2.2.5 OpenAlex Publication Database Finally, w..."
9,10,text,LLM interactions. This is the sample we will f...


Document Modality


modality
text     42
image    14
Name: count, dtype: int64

In [44]:
## Create Embeddings

embeddings = HuggingFaceEmbeddings(
    model_name = EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings" : True}
)

EMBEDDING_DIMENSION = len(
    embeddings.embed_query("dimension check")
)

print("Embedding Model: ", EMBEDDING_MODEL)
print("Embedding dimension: ", EMBEDDING_DIMENSION)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19202.30it/s]


Embedding Model:  sentence-transformers/all-MiniLM-L6-v2
Embedding dimension:  384
